# Initial population from a function of parameters

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `detailed/InitialPopulationGraphobject` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

summer2's graph-object initial population becomes a `Split(..., by=)` callable
that returns a table of weights from parameters.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
import jax.numpy as jnp

from summer4 import (
    Compartments,
    Everything,
    InitialPopulation,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Split,
)
from summer4.epi import EpiModel

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("S", "I", "R"))
age = Property("age", ("young", "old"))
loc = Property("loc", ("WA", "other"))
imm = Property("imm", ("yes", "no"))
pmap = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(loc)
    .stratify(imm)
)

# Absolute base totals by age × location (S only); imm split by age via callable.
base = {
    state["S"] & age["young"] & loc["WA"]: 1000.0,
    state["S"] & age["old"] & loc["WA"]: 2000.0,
    state["S"] & age["young"] & loc["other"]: 10000.0,
    state["S"] & age["old"] & loc["other"]: 30000.0,
    state["I"]: 10.0,
}


def imm_table(p):
    return jnp.array(
        [
            [p["vacc_young"], 1.0 - p["vacc_young"]],
            [p["vacc_old"], 1.0 - p["vacc_old"]],
        ]
    )


plan = InitialPopulation(base, splits=(Split(imm, imm_table, by=(age,)),)).compile(pmap)
params = {"vacc_young": 0.2, "vacc_old": 0.6}
y0 = plan.evaluate(params)
data = np.asarray(y0.data)
assert np.isclose(float(data.sum()), 43010.0)
young_wa_yes = float(data[pmap.select(state["S"] & age["young"] & loc["WA"] & imm["yes"])][0])
assert np.isclose(young_wa_yes, 1000.0 * 0.2)

epi = EpiModel(pmap, infectious=state["I"])
epi.set_mixing_matrix(age, np.eye(2), check_reciprocal=False)
epi.add_infection_frequency_flow("infection", state["S"], state["I"], 0.3)
epi.add_transition_flow("recovery", state["I"], state["R"], 0.1)
epi.set_initial_population(base, splits=(Split(imm, imm_table, by=(age,)),))
cm = epi.compile()
save = SavePlan(requests={"comp": SaveRequest(Compartments())})
res = cm.run(params, t0=0.0, t1=50.0, dt=1.0, save=save, solver="euler")
res["comp"].to_pandas().plot(title="Initial population from by= callable")